# Analysis workflow tests

Loads the survey and sensor data, builds a panel dataset, and converts it for two kinds of analysis:

- [SurvivalScale](https://github.com/tazzben/Likert-estimation) (section 5), for Likert-scale responses as the dependent variable.
- OLS with [statsmodels](https://www.statsmodels.org/) (section 6), for a continuous dependent variable such as age or a sensor measure.

Each section ends with assertions that fail loudly if something is wrong.

Install the estimators with `pip install SurvivalScale statsmodels` to run the final sections.

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path
import sys

for project_root in (Path.cwd(), *Path.cwd().parents):
    src_dir = project_root / "src"
    if (src_dir / "CBACABLab").is_dir():
        sys.path.insert(0, str(src_dir))
        break

import CBACABLab

DATABASE = "data.db"
SURVEY_FILE = "Survey.xlsx"
SENSOR_FOLDER = "./sensorData/"
REBUILD = True  # reload the survey and sensor data; set False to reuse the existing database

# Questions answered by every respondent, used as controls
RESPONDENT_QUESTIONS = ["Age", "Gender"]

# Questions with a Likert-style response, and the scale for each.
# A scale is a name from settings.json (see CBACABLab.ListLikertScales()) or a {label: k} mapping.
LIKERT_SCALES = {
    "B5-1Pre": {"Less than $102": 0, "Exactly $102": 1, "More than $102": 2},
    "B5-2Pre": {"Less than today": 0, "Exactly the same": 1, "More than today": 2},
}

TAGS = ["left", "right"]  # e.g. ["left", "right"] to keep only some sensor tags
FACE_PROPERTIES = ["Attention_left", "Attention_right"] # e.g. ["Joy", "Attention"]

## 1. Load the survey and sensor data

In [2]:
if REBUILD:
    CBACABLab.LoadSurveyData(databaseName=DATABASE, surveyName=SURVEY_FILE)
    CBACABLab.LoadSensorData(
        databaseName=DATABASE,
        sensorFolder=SENSOR_FOLDER,
        boundedboxes=[(0, 960, 0, 1080)],
        tag="left",
        rebuild=True,
    )
    CBACABLab.LoadSensorData(
        databaseName=DATABASE,
        sensorFolder=SENSOR_FOLDER,
        boundedboxes=[(960, 1920, 0, 1080)],
        tag="right",
    )

In [3]:
# Test: the tables exist and the survey has data
with sqlite3.connect(DATABASE) as con:
    counts = {
        table: con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        for table in ["FullSurvey", "SensorData", "FaceData"]
    }
print(counts)
assert counts["FullSurvey"] > 0, "FullSurvey is empty"
if counts["SensorData"] == 0:
    print("WARNING: SensorData is empty. Sensor columns will be NaN; check the sensor folder and respondent ids.")

{'FullSurvey': 6496, 'SensorData': 60, 'FaceData': 1892}


In [4]:
# Test: every question we plan to use is in the survey
available = set(CBACABLab.ListQuestions(DATABASE))
wanted = set(RESPONDENT_QUESTIONS) | set(LIKERT_SCALES)
missing = wanted - available
assert not missing, f"Questions not found in the survey: {sorted(missing)}"

## 2. Build the panel dataset

In [5]:
panel = CBACABLab.CreatePanelData(
    question_level_columns=list(LIKERT_SCALES),
    respondent_level_columns=RESPONDENT_QUESTIONS,
    tagFilter=TAGS,
    faceProperties=FACE_PROPERTIES,
    databaseName=DATABASE,
)
panel

,respondent_id,question,response,tag,pupil,duration,property,value,Age,Gender
0,18394,B5-1Pre,More than $102,NaN,NaN,NaN,NaN,NaN,27,Female
1,18394,B5-2Pre,Less than today,NaN,NaN,NaN,NaN,NaN,27,Female
2,27853,B5-1Pre,More than $102,NaN,NaN,NaN,NaN,NaN,23,Female
3,27853,B5-2Pre,Don’t know,NaN,NaN,NaN,NaN,NaN,23,Female
4,28522,B5-1Pre,More than $102,NaN,NaN,NaN,NaN,NaN,21,Female
...,...,...,...,...,...,...,...,...,...,...
107,32047,B5-2Pre,Less than today,NaN,NaN,NaN,NaN,NaN,19,Male
108,32065,B5-1Pre,More than $102,NaN,NaN,NaN,NaN,NaN,19,Male
109,32065,B5-2Pre,Less than today,NaN,NaN,NaN,NaN,NaN,19,Male
110,32113,B5-1Pre,More than $102,NaN,NaN,NaN,NaN,NaN,19,Male


In [6]:
# Tests: panel structure
required = {"respondent_id", "question", "response", *RESPONDENT_QUESTIONS}
assert required <= set(panel.columns), f"Missing columns: {required - set(panel.columns)}"
assert set(panel["question"]) == set(LIKERT_SCALES), "Panel contains unexpected questions"

# Strictly one row per respondent and question
assert not panel.duplicated(["respondent_id", "question"]).any(), "More than one row per respondent and question"

# Every survey response is kept, even without sensor data (left joins)
with sqlite3.connect(DATABASE) as con:
    placeholders = ", ".join("?" * len(LIKERT_SCALES))
    survey_count = con.execute(
        f"SELECT COUNT(*) FROM FullSurvey WHERE question IN ({placeholders})", list(LIKERT_SCALES)
    ).fetchone()[0]
assert len(panel) == survey_count, "Survey responses were lost"

## 3. Convert responses to `k` and `bound`

Responses outside a question's scale (for example "Don't know") can't be converted, so they are listed and dropped first.

In [7]:
def labels(scale):
    if isinstance(scale, str):
        scale = dict(CBACABLab.ListLikertScales(show_values=True))[scale]
    return {str(label).strip().lower() for label in scale}

in_scale = pd.Series(
    [str(r).strip().lower() in labels(LIKERT_SCALES[q]) for q, r in zip(panel["question"], panel["response"])],
    index=panel.index,
)
print("Dropping responses outside their scale:")
print(panel.loc[~in_scale, ["question", "response"]].value_counts())
panel_likert = CBACABLab.AddLikertColumns(panel[in_scale], LIKERT_SCALES)
panel_likert[["respondent_id", "question", "response", "k", "bound"]].drop_duplicates().head(10)

Dropping responses outside their scale:
question  response  
B5-2Pre   Don’t know    2
Name: count, dtype: int64


,respondent_id,question,response,k,bound
0,18394,B5-1Pre,More than $102,2,2
1,18394,B5-2Pre,Less than today,0,2
2,27853,B5-1Pre,More than $102,2,2
4,28522,B5-1Pre,More than $102,2,2
5,28522,B5-2Pre,Less than today,0,2
6,28783,B5-1Pre,More than $102,2,2
7,28783,B5-2Pre,Less than today,0,2
8,28807,B5-1Pre,More than $102,2,2
9,28807,B5-2Pre,Less than today,0,2
10,29269,B5-1Pre,More than $102,2,2


In [8]:
# Tests: k and bound
assert panel_likert[["k", "bound"]].notna().all().all(), "k or bound has missing values"
assert (panel_likert["k"] >= 0).all() and (panel_likert["k"] <= panel_likert["bound"]).all()

for question, scale in LIKERT_SCALES.items():
    expected_bound = max(scale.values()) if isinstance(scale, dict) else None
    rows = panel_likert[panel_likert["question"] == question]
    assert rows["bound"].nunique() == 1, f"{question}: bound should be constant"
    if expected_bound is not None:
        assert (rows["bound"] == expected_bound).all(), f"{question}: wrong bound"
    # The same response must always map to the same k
    assert rows.groupby("response")["k"].nunique().max() == 1, f"{question}: inconsistent mapping"

# A response not in a question's scale must raise
try:
    CBACABLab.AddLikertColumns(pd.DataFrame({"question": ["Q"], "response": ["nonsense"]}), {"Q": "agree5"})
    raise AssertionError("Unknown response did not raise")
except ValueError:
    pass

# A question without a scale must raise
try:
    CBACABLab.AddLikertColumns(pd.DataFrame({"question": ["Q"], "response": ["Agree"]}), {"Other": "agree5"})
    raise AssertionError("Missing scale did not raise")
except KeyError:
    pass

In [9]:
# Test: the scales shipped in settings.json convert correctly
scales = dict(CBACABLab.ListLikertScales(show_values=True))
assert {"agree5", "agree7", "satisfied5", "quality5"} <= set(scales)
demo = pd.DataFrame({
    "question": ["A", "A", "B", "B"],
    "response": ["Strongly disagree", "Strongly agree", "Very poor", "Excellent"],
})
result = CBACABLab.AddLikertColumns(demo, {"A": "agree5", "B": "quality5"})
assert result["k"].tolist() == [0, 4, 0, 4]
assert result["bound"].tolist() == [4, 4, 4, 4]

# Different scales per question give different bounds
result = CBACABLab.AddLikertColumns(demo, {"A": "agree5", "B": "quality7"})
assert result["bound"].tolist() == [4, 4, 6, 6]

## 4. Prepare the data for SurvivalScale

SurvivalScale expects one row per observation with `id`, `block`, `question`, numeric feature columns, `k` and `bound`. Here `block` is the respondent, so errors can be clustered by respondent.

In [10]:
base = panel_likert.reset_index(drop=True)

FEATURES = ["Age", "Female"]
survival = pd.DataFrame({
    "id":  base["respondent_id"],
    "block": base["respondent_id"],
    "question": base["question"].astype("category").cat.codes + 1,
    "Age": pd.to_numeric(base["Age"], errors="coerce"),
    "Female": (base["Gender"] == "Female").astype(int),
    "k": base["k"].astype(int),
    "bound": base["bound"].astype(int),
}).dropna()
survival.head()

,id,block,question,Age,Female,k,bound
0,18394,18394,1,27,1,2,2
1,18394,18394,2,27,1,0,2
2,27853,27853,1,23,1,2,2
3,28522,28522,1,21,1,2,2
4,28522,28522,2,21,1,0,2


In [11]:
# Tests: SurvivalScale input
required = {"id", "block", "question", "k", "bound", *FEATURES}
assert required <= set(survival.columns)
assert not survival.duplicated(["block", "question"]).any(), "More than one row per respondent and question"
assert all(pd.api.types.is_numeric_dtype(survival[c]) for c in survival.columns)
assert survival.notna().all().all()
assert (survival["k"] <= survival["bound"]).all()
print(survival.shape)

(110, 7)


## 5. Run the estimator

In [12]:

%pip install SurvivalScale 


from SurvivalScale import get_results

if get_results is not None:
    cj, beta, metrics = get_results(
        survival,
        bootstrap_iterations=100,
        alpha=0.05,
        columns=FEATURES,
        block_id="block",
    )
    assert cj is not None and beta is not None and metrics is not None
    display(beta)

Note: you may need to restart the kernel to use updated packages.
Performing bootstrap analysis to estimate confidence intervals and p-values...


/opt/anaconda3/lib/python3.14/site-packages/scipy/optimize/_optimize.py:2540: RuntimeWarning: invalid value encountered in scalar multiply
  tmp2 = (x - v) * (fx - fw)
  0%|          | 0/100 [00:00<?, ?it/s]/opt/anaconda3/lib/python3.14/site-packages/scipy/optimize/_optimize.py:2540: RuntimeWarning: invalid value encountered in scalar multiply
  tmp2 = (x - v) * (fx - fw)
/opt/anaconda3/lib/python3.14/site-packages/scipy/optimize/_optimize.py:2540: RuntimeWarning: invalid value encountered in scalar multiply
  tmp2 = (x - v) * (fx - fw)
/opt/anaconda3/lib/python3.14/site-packages/scipy/optimize/_optimize.py:2540: RuntimeWarning: invalid value encountered in scalar multiply
  tmp2 = (x - v) * (fx - fw)
/opt/anaconda3/lib/python3.14/site-packages/scipy/optimize/_optimize.py:2540: RuntimeWarning: invalid value encountered in scalar multiply
  tmp2 = (x - v) * (fx - fw)
/opt/anaconda3/lib/python3.14/site-packages/scipy/optimize/_optimize.py:2540: RuntimeWarning: invalid value encountered i

,beta,marginal,marginal_discrete,lower_bound,upper_bound,p_value
Age,-0.083506,-0.009373,-0.020863,-0.175738,-0.062605,0.00
Female,-0.003727,-0.000418,-0.000418,-0.559101,1.295306,0.92


## 6. OLS with statsmodels

OLS needs a continuous outcome. Likert responses are ordinal, so here the continuous variable is `Age`, used as the dependent variable, with the Likert response `k` as a regressor. Question fixed effects control for the different scales, and standard errors are clustered by respondent because each respondent appears once per question.

In [ ]:
%pip install statsmodels
import statsmodels.formula.api as smf

ols_data = survival.copy()
ols_data["question"] = ols_data["question"].astype("category")

ols_model = smf.ols("Age ~ k + Female + C(question)", data=ols_data).fit(
    cov_type="cluster", cov_kwds={"groups": ols_data["block"]}
)
print(ols_model.summary())

In [ ]:
# Tests: OLS ran on the full sample with the expected terms
assert int(ols_model.nobs) == len(ols_data)
assert {"k", "Female"} <= set(ols_model.params.index)
assert ols_model.params.notna().all()

### OLS with a continuous sensor outcome

When sensor data is loaded, any pupil or duration column from the panel can be the dependent variable. This cell uses the first one that has data and is skipped otherwise.

In [ ]:
sensor_columns = [
    c for c in panel_likert.columns
    if c.startswith(("pupil", "duration")) and panel_likert[c].notna().any()
]
if not sensor_columns:
    print("No sensor data in the panel; skipping.")
else:
    outcome = sensor_columns[0]
    sensor_data = pd.DataFrame({
        "outcome": panel_likert[outcome],
        "k": panel_likert["k"].astype(int),
        "Age": pd.to_numeric(panel_likert["Age"], errors="coerce"),
        "Female": (panel_likert["Gender"] == "Female").astype(int),
        "question": panel_likert["question"],
        "block": panel_likert["respondent_id"],
    }).dropna()
    sensor_model = smf.ols("outcome ~ k + Age + Female + C(question)", data=sensor_data).fit(
        cov_type="cluster", cov_kwds={"groups": sensor_data["block"]}
    )
    print(f"Outcome: {outcome}")
    print(sensor_model.summary())
    assert int(sensor_model.nobs) == len(sensor_data)

No sensor data in the panel; skipping.
